# 02 — Local, Encoder-Based Knowledge Graph Extraction

**From Clinical Case Reports to Knowledge Graphs**

This notebook implements the pipeline proposed in
[`docs/kg_extraction_methodology.md`](../docs/kg_extraction_methodology.md):
an illustrative, best-achievable-under-local-constraints KG extraction over a
configurable sample of MultiCaRe clinical cases, using only local,
encoder-based (BERT-family) models — no remote LLM APIs, everything runs on
this machine.

**Prerequisites**
- `01_data_preparation.ipynb` has already been run, so `../data/clinical_cases.duckdb`
  exists with `cases` and `metadata` tables. This notebook only *reads* that
  database — it never downloads anything itself.
- Run inside the `env/kg-extraction` environment (Docker or `uv` — see
  `../README.md`). The first run downloads several encoder models (~2GB
  combined); every run after that is fully offline.

**What this is not:** a production pipeline. It's a laptop-scale
illustration meant to show students "how far NLP can go" before they build
the same kind of knowledge graph by hand, step by step, in
`Activity_Plan_Clinical_Cases_to_Knowledge_Graphs.md`. See Section 5 of the
methodology doc for the honest quality trade-offs against a generative-LLM
approach, and Section 2 for the full stage-by-stage design this notebook
follows.

## 1. Configuration

Every parameter that controls this run lives here — nothing else in the
notebook should need editing to change the sample size, models, or
thresholds.

In [1]:
from pathlib import Path

# --- Sample --------------------------------------------------------------
SAMPLE_SIZE = 50            # number of clinical cases to process end-to-end — see Section 10
                             # for a runtime estimate before raising this
MIN_CASE_CHARS = 200        # drop degenerate/too-short cases before sampling
RANDOM_SEED = 42

# --- Paths -----------------------------------------------------------------
DB_PATH = Path("../data/clinical_cases.duckdb")
OUTPUT_DIR = Path("../data/kg_extraction")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Models (all local/encoder-based — see docs/kg_extraction_methodology.md) ---
NER_MODEL_NAME = "d4data/biomedical-ner-all"
SAPBERT_MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
ZEROSHOT_MODEL_NAME = "MoritzLaurer/deberta-v3-base-zeroshot-v2.0"
SCISPACY_MODEL = "en_core_sci_lg"

# --- Feature toggles (best-effort: the pipeline still runs if unavailable) ---
USE_UMLS_LINKING = True    # requires scispacy + nmslib; see README "known friction point"
USE_NEGATION = True        # requires negspacy

# --- Thresholds --------------------------------------------------------------
NER_SCORE_THRESHOLD = 0.5          # minimum HF NER confidence to keep a span
RELATION_TYPE_THRESHOLD = 0.5      # minimum zero-shot score to accept a relation label (Stage 5b)
FAITHFULNESS_THRESHOLD = 0.6       # stricter re-check on the same score (Stage 7)
ENTITY_RESOLUTION_SIM_THRESHOLD = 0.85   # SapBERT cosine similarity to merge two ungrounded entities (Stage 6)
UMLS_LINKING_SCORE_THRESHOLD = 0.75      # minimum linker confidence to accept a CUI
MAX_RELATION_CANDIDATES_PER_CASE = 30    # safety cap for Stage 5 — case_text ranges from 9 to
                                          # 79,243 chars in this corpus; without this, one long
                                          # outlier case could dominate the whole run (Section 9)

# --- Compute -----------------------------------------------------------------
DEVICE = "cpu"   # "cuda" or "mps" if you have a usable GPU; CPU is the tested default
HF_BATCH_SIZE = 16

assert FAITHFULNESS_THRESHOLD >= RELATION_TYPE_THRESHOLD, (
    "the faithfulness filter (Stage 7) reuses Stage 5b's score, so it should "
    "be at least as strict as the typing threshold"
)

## 2. Setup

`scispacy` and `negspacy` are treated as optional throughout this notebook.
If either of them (or a native dependency, e.g. `nmslib`) isn't installed,
a warning is printed and the pipeline keeps going with reduced functionality
(no UMLS grounding and/or no negation tagging) instead of failing outright —
see `../README.md` for the known `nmslib` build friction on some platforms.

In [2]:
import os
import random
import warnings
from itertools import combinations

import duckdb
import numpy as np
import pandas as pd
import networkx as nx
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer
from transformers import pipeline as hf_pipeline
from sklearn.cluster import AgglomerativeClustering

import spacy

try:
    import scispacy  # noqa: F401  (registers scispaCy's spaCy factories on import)
    from scispacy.linking import EntityLinker  # noqa: F401
    SCISPACY_AVAILABLE = True
except ImportError:
    SCISPACY_AVAILABLE = False
    print("scispacy not installed — UMLS grounding will be skipped.")

try:
    import negspacy  # noqa: F401
    from negspacy.negation import Negex  # noqa: F401  (registers the "negex" factory)
    NEGSPACY_AVAILABLE = True
except ImportError:
    NEGSPACY_AVAILABLE = False
    print("negspacy not installed — negation tagging will be skipped.")

warnings.filterwarnings("ignore", category=FutureWarning)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DEVICE_ID = {"cpu": -1, "cuda": 0, "mps": 0}.get(DEVICE, -1)
TORCH_DEVICE = torch.device(DEVICE if DEVICE != "cpu" else "cpu")

/home/santanche/git/2learn/nlp2learn/to-kg/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Your CPU supports instructions that this binary was not compiled to use: SSE3 SSE4.1 SSE4.2 AVX AVX2
For maximum performance, you can install NMSLIB from sources 
pip install --no-binary :all: nmslib


## 3. Load the sample from DuckDB

Connects to the database built by `01_data_preparation.ipynb` (read-only —
this notebook never writes to it) and draws a sample of `SAMPLE_SIZE` cases,
stratified by gender and a coarse age bracket so the resulting graph isn't
dominated by one demographic slice (a proxy for the specialty/age/gender
diversity goal in Section 4 of the methodology doc). Falls back to a plain
random sample once `SAMPLE_SIZE` approaches the size of the filtered pool.

In [3]:
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"{DB_PATH} not found — run 01_data_preparation.ipynb first "
        "(this notebook only reads the database, it does not build it)."
    )

con = duckdb.connect(str(DB_PATH), read_only=True)
cases_df = con.execute(
    """
    SELECT c.case_id, c.article_id, c.case_text, c.age, c.gender
    FROM cases AS c
    WHERE LENGTH(c.case_text) >= ?
    """,
    [MIN_CASE_CHARS],
).df()
con.close()

print(f"{len(cases_df):,} cases available after the {MIN_CASE_CHARS}-char minimum-length filter.")

98,147 cases available after the 200-char minimum-length filter.


In [4]:
def age_bucket(age):
    if pd.isna(age):
        return "unknown"
    if age < 18:
        return "child"
    if age < 65:
        return "adult"
    return "senior"


def stratified_sample(df, n, seed):
    if n >= len(df):
        return df.sample(frac=1, random_state=seed).reset_index(drop=True)

    strata = df["gender"].fillna("Unknown") + "_" + df["age"].apply(age_bucket)
    target_counts = (strata.value_counts(normalize=True) * n).round().astype(int)

    parts = []
    for stratum, count in target_counts.items():
        pool = df[strata == stratum]
        count = min(count, len(pool))
        if count > 0:
            parts.append(pool.sample(n=count, random_state=seed))

    sampled = pd.concat(parts).drop_duplicates(subset="case_id")

    # Rounding can leave the total a little short of n; top up from the rest.
    shortfall = n - len(sampled)
    if shortfall > 0:
        remaining = df[~df["case_id"].isin(sampled["case_id"])]
        top_up = remaining.sample(n=min(shortfall, len(remaining)), random_state=seed)
        sampled = pd.concat([sampled, top_up])

    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)


sample_df = stratified_sample(cases_df, SAMPLE_SIZE, RANDOM_SEED)
print(f"Sampled {len(sample_df)} cases.")
sample_df[["case_id", "age", "gender"]].head()

Sampled 50 cases.


,case_id,age,gender
0,PMC2842968_02,24.0,Female
1,PMC2796237_01,3.0,Female
2,PMC9841794_02,84.0,Male
3,PMC10948227_06,NaN,Unknown
4,PMC1201178_01,49.0,Female


## 4. Build the local NLP pipeline (segmentation, dependency parse, negation, UMLS linking)

One scispaCy pipeline, built once and reused for Stages 1, 3, 4, and 5a.
Its own NER component is excluded (`exclude=["ner"]`) — entity spans come
from the dedicated biomedical NER model in Section 5 instead; this pipeline
only contributes sentence segmentation, dependency parsing, negation, and
UMLS grounding on top of externally-supplied entity spans.

In [5]:
def build_scispacy_pipeline():
    if not SCISPACY_AVAILABLE:
        return None, False

    nlp = spacy.load(SCISPACY_MODEL, exclude=["ner"])

    if USE_NEGATION and NEGSPACY_AVAILABLE:
        nlp.add_pipe("negex")

    linker_available = False
    if USE_UMLS_LINKING:
        try:
            nlp.add_pipe(
                "scispacy_linker",
                config={
                    "resolve_abbreviations": True,
                    "linker_name": "umls",
                    "max_entities_per_mention": 1,
                    "threshold": UMLS_LINKING_SCORE_THRESHOLD,
                },
            )
            linker_available = True
        except Exception as exc:
            print(f"UMLS linker unavailable ({exc!r}) — continuing without CUI grounding.")

    return nlp, linker_available


nlp, umls_linker_available = build_scispacy_pipeline()

if nlp is None:
    print("Running without scispaCy: no dependency parse, negation, or UMLS grounding.")
else:
    print(f"scispaCy pipeline ready. UMLS linking: {'on' if umls_linker_available else 'off'}.")

/home/santanche/git/2learn/nlp2learn/to-kg/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.1.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/santanche/git/2learn/nlp2learn/to-kg/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.1.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


scispaCy pipeline ready. UMLS linking: on.


## 5. Load the encoder models

Three models, all encoder-only:

- **NER** — `d4data/biomedical-ner-all` (DistilBERT, 66M params, trained on
  MACCROBAT clinical case reports — a close genre match to `case_text`).
  Stage 2.
- **Zero-shot NLI** — `MoritzLaurer/deberta-v3-base-zeroshot-v2.0`
  (DeBERTa-v3-base, ~0.2B params). Reused for *both* Stage 5b (relation
  typing) and Stage 7 (faithfulness filter) — the same entailment score
  read at two different thresholds, not two separate models.
- **SapBERT** — `cambridgeltl/SapBERT-from-PubMedBERT-fulltext`. Entity
  mention embeddings for Stage 6 (cross-case entity resolution).

First run downloads ~2GB combined from the Hugging Face hub; cached locally
afterward — no network access needed on subsequent runs.

In [6]:
ner_pipeline = hf_pipeline(
    "token-classification",
    model=NER_MODEL_NAME,
    aggregation_strategy="simple",
    device=DEVICE_ID,
)

zeroshot_pipeline = hf_pipeline(
    "zero-shot-classification",
    model=ZEROSHOT_MODEL_NAME,
    device=DEVICE_ID,
)

sapbert_tokenizer = AutoTokenizer.from_pretrained(SAPBERT_MODEL_NAME)
sapbert_model = AutoModel.from_pretrained(SAPBERT_MODEL_NAME).to(TORCH_DEVICE)
sapbert_model.eval()


def embed_texts(texts, batch_size=HF_BATCH_SIZE):
    """SapBERT's recommended pooling: the [CLS] token of the last hidden state."""
    if not texts:
        return np.empty((0, sapbert_model.config.hidden_size))
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = sapbert_tokenizer(
            batch, padding=True, truncation=True, max_length=64, return_tensors="pt"
        ).to(TORCH_DEVICE)
        with torch.no_grad():
            output = sapbert_model(**encoded)
        cls_embeddings = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_embeddings)
    return np.vstack(all_embeddings)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2318.74it/s]


## 6. Entity schema: mapping model labels to the activity's categories

`d4data/biomedical-ner-all` outputs 15 entity types (BIO-tagged); only some
map onto the six categories from Step 3 of the activity plan. The rest
(`Dosage`, `Severity`, `Duration`, `Frequency`, `Family_history`, `History`,
…) are genuinely useful qualifiers, but attaching each one to the *specific*
entity it modifies is itself a small relation-extraction problem — out of
scope here, so they're kept in a separate table (`qualifiers_df`, Section
10) for inspection rather than silently dropped or naively attached.

This cell also defines `PLAUSIBLE_RELATIONS`: which relation labels are
even sensible between a given pair of entity *categories* (e.g. only
`located_in` makes sense for an Anatomy/Symptom pair). Section 9 uses it to
avoid asking the zero-shot model to score all ten relation labels for
every candidate — the single biggest performance fix in this notebook.

In [7]:
LABEL_MAP = {
    "Sign_symptom": "Symptom",
    "Disease_disorder": "Disease",
    "Medication": "Drug",
    "Lab_value": "Laboratory Test",
    "Biological_structure": "Anatomy",
    "Diagnostic_procedure": "Procedure",
    "Therapeutic_procedure": "Procedure",
}

QUALIFIER_LABELS = {
    "Dosage", "Severity", "Duration", "Frequency", "Family_history",
    "History", "Administration", "Outcome", "Clinical_event",
}

RELATION_LABELS = [
    "has_symptom", "treats", "reveals", "diagnosed_with", "underwent",
    "located_in", "administered_at_dose", "has_lab_result", "ruled_out",
    "family_history_of",
]

RELATION_CUE_LEMMAS = {
    "treat", "administer", "prescribe", "give", "start",
    "reveal", "show", "demonstrate", "confirm", "indicate",
    "diagnose", "undergo", "perform", "elevate", "detect",
}

# Which relations are plausible between a given *pair of entity categories*
# — e.g. scoring "administered_at_dose" for an Anatomy/Symptom pair is both
# nonsensical and wasted compute. Keyed by the unordered category pair
# (text order doesn't reliably indicate semantic subject/object direction);
# a missing entry means every candidate for that pair is dropped before any
# model call. Measured impact: without this restriction, the zero-shot
# model scores all 10 RELATION_LABELS for every candidate (~8s/candidate on
# one CPU-only laptop); with it, most candidates score only 1-3 labels —
# roughly a 10x reduction in Stage 5b's cost.
PLAUSIBLE_RELATIONS = {
    ("Disease", "Symptom"): ["has_symptom", "ruled_out"],
    ("Disease", "Drug"): ["treats", "ruled_out"],
    ("Disease", "Procedure"): ["reveals", "diagnosed_with", "ruled_out"],
    ("Disease", "Laboratory Test"): ["has_lab_result", "diagnosed_with", "ruled_out"],
    ("Anatomy", "Disease"): ["located_in"],
    ("Disease", "Disease"): ["family_history_of", "ruled_out"],
    ("Procedure", "Symptom"): ["reveals"],
    ("Laboratory Test", "Symptom"): ["has_lab_result"],
    ("Anatomy", "Symptom"): ["located_in"],
    ("Drug", "Symptom"): ["treats"],
    ("Symptom", "Symptom"): ["family_history_of"],
    ("Drug", "Procedure"): ["administered_at_dose", "underwent"],
    ("Anatomy", "Drug"): ["located_in", "administered_at_dose"],
    ("Laboratory Test", "Procedure"): ["reveals", "has_lab_result"],
    ("Anatomy", "Procedure"): ["located_in", "underwent"],
    ("Anatomy", "Laboratory Test"): ["located_in", "has_lab_result"],
}


def plausible_labels(category_a, category_b):
    return PLAUSIBLE_RELATIONS.get(tuple(sorted((category_a, category_b))), [])

## 7. Stage 2 — Candidate NER

Runs the DistilBERT NER model over each case, keeps only spans mapped to one
of the six activity categories above `NER_SCORE_THRESHOLD`, and records
qualifier spans (Section 6) separately.

In [8]:
def run_ner(text):
    raw_spans = ner_pipeline(text)
    entities, qualifiers = [], []
    for span in raw_spans:
        if span["score"] < NER_SCORE_THRESHOLD:
            continue
        native_label = span["entity_group"]
        if native_label in LABEL_MAP:
            entities.append({
                "text": span["word"],
                "start": span["start"],
                "end": span["end"],
                "native_label": native_label,
                "category": LABEL_MAP[native_label],
                "ner_score": float(span["score"]),
            })
        elif native_label in QUALIFIER_LABELS:
            qualifiers.append({
                "text": span["word"],
                "start": span["start"],
                "end": span["end"],
                "native_label": native_label,
                "ner_score": float(span["score"]),
            })
    return entities, qualifiers

## 8. Stage 3 + 4 — Grounding and negation

Reuses one scispaCy `Doc` per case for both stages: the NER spans from
Stage 2 are injected into `doc.ents` (scispaCy's own NER was excluded in
Section 4 precisely so it doesn't fight with these externally-supplied
spans), then `negex` and the UMLS linker — both of which operate on
`doc.ents` — run over that same `Doc`. Grounding is best-effort: if the
UMLS linker isn't available, entities simply keep `cui = None` and Stage 6
falls back to embedding-only resolution for all of them.

In [9]:
def annotate_entities(case_text, entities):
    if nlp is None:
        annotated = [
            {**e, "start_token": None, "end_token": None, "negated": None,
             "cui": None, "cui_score": None, "canonical_name": None}
            for e in entities
        ]
        return annotated, None

    doc = nlp(case_text)
    if not entities:
        return [], doc

    ent_spans, span_by_start = [], {}
    for e in entities:
        # `label=` is required here — char_span() without one returns an
        # unlabeled span, and spaCy silently drops unlabeled spans when
        # assigned to doc.ents (no error, just an empty result).
        span = doc.char_span(e["start"], e["end"], label=e["category"], alignment_mode="expand")
        if span is None:
            continue
        ent_spans.append(span)
        span_by_start[span.start_char] = e

    try:
        doc.ents = ent_spans
    except ValueError:
        # Overlapping spans after alignment_mode="expand" — keep the longer one per overlap.
        ent_spans = sorted(ent_spans, key=lambda s: s.end_char - s.start_char, reverse=True)
        kept, occupied = [], set()
        for span in ent_spans:
            token_range = set(range(span.start, span.end))
            if token_range & occupied:
                continue
            kept.append(span)
            occupied |= token_range
        doc.ents = kept

    if USE_NEGATION and NEGSPACY_AVAILABLE and "negex" in nlp.pipe_names:
        doc = nlp.get_pipe("negex")(doc)

    linker = None
    if umls_linker_available:
        linker = nlp.get_pipe("scispacy_linker")
        doc = linker(doc)

    annotated = []
    for ent in doc.ents:
        base = span_by_start.get(ent.start_char)
        if base is None:
            continue
        cui, cui_score, canonical_name = None, None, None
        if linker is not None and ent._.kb_ents:
            cui, cui_score = ent._.kb_ents[0]
            canonical_name = linker.kb.cui_to_entity[cui].canonical_name
        annotated.append({
            **base,
            "start_token": ent.start,
            "end_token": ent.end,
            "negated": bool(ent._.negex) if (USE_NEGATION and NEGSPACY_AVAILABLE) else None,
            "cui": cui,
            "cui_score": float(cui_score) if cui_score is not None else None,
            "canonical_name": canonical_name,
            "spacy_span": ent,
        })
    return annotated, doc

## 9. Stage 5 — Relation candidate generation and typing

**5a — candidate generation (rule-based, not a model):** within each
sentence, an entity pair is considered a relation candidate only if a cue
lemma (Section 6) lies on the shortest dependency path between them — a
lightweight stand-in for full semantic role labeling, using scispaCy's
dependency parse. Candidates are then dropped unless `PLAUSIBLE_RELATIONS`
(Section 6) actually lists a plausible relation for that pair of entity
*categories*, and the survivors are capped at
`MAX_RELATION_CANDIDATES_PER_CASE` — a safety valve, since `case_text`
ranges from 9 to 79,243 characters in this corpus and one outlier case
could otherwise dominate the whole run.

**5b — typing (encoder-based):** each surviving candidate is scored
against *only its plausible labels* (not all ten) using the zero-shot NLI
model, with the actual subject/object text spliced into the hypothesis
template (`"{subject} {} {object}."`) — so the model scores concrete
sentences like *"Azithromycin treats pneumonia."*, not an abstract label.

**Why this matters for runtime:** an earlier version of this notebook
scored all 10 `RELATION_LABELS` for every candidate — measured at ~8s per
candidate on one CPU-only laptop, and a single ~1,800-character case
generated 25+ candidates, i.e. minutes per case and hours for
`SAMPLE_SIZE = 200`. Restricting to plausible labels cut the same case's
Stage 5 cost from ~197s to ~15s. The top-scoring relation is kept if it
clears `RELATION_TYPE_THRESHOLD`. **Stage 7 reuses this same score** at a
stricter `FAITHFULNESS_THRESHOLD` later (Section 12) — no second model
call needed.

In [10]:
def generate_relation_candidates(doc, entities):
    if doc is None or len(entities) < 2:
        return []
    candidates = []
    for sent in doc.sents:
        # Require the *whole* entity span inside the sentence, not just its
        # start — an entity whose span was expanded (Section 8) can still
        # straddle a sentence boundary (e.g. a lab value like "1.100 x 10^9/L"
        # split across scispaCy's sentence segmentation). A start-only check
        # lets such an entity's root token land in a different sentence's
        # dependency tree, which isn't in `graph` below.
        sent_entities = [
            e for e in entities
            if sent.start <= e["start_token"] and e["end_token"] <= sent.end
        ]
        if len(sent_entities) < 2:
            continue
        graph = nx.Graph()
        for token in sent:
            graph.add_edge(token.i, token.head.i)
        for e1, e2 in combinations(sent_entities, 2):
            try:
                path = nx.shortest_path(graph, e1["spacy_span"].root.i, e2["spacy_span"].root.i)
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                # NodeNotFound is a defensive fallback, not the primary fix —
                # the containment check above should already prevent it, but
                # this notebook runs unattended over many cases, so a single
                # unforeseen edge case should skip that candidate, not crash
                # the whole run.
                continue
            path_lemmas = {doc[i].lemma_.lower() for i in path}
            if not (path_lemmas & RELATION_CUE_LEMMAS):
                continue
            labels = plausible_labels(e1["category"], e2["category"])
            if not labels:
                continue
            candidates.append((e1, e2, sent.text, labels))
    return candidates[:MAX_RELATION_CANDIDATES_PER_CASE]


def type_relation(sentence, subject_text, object_text, labels):
    hypothesis_template = f"{subject_text} {{}} {object_text}."
    result = zeroshot_pipeline(
        sentence, candidate_labels=labels,
        hypothesis_template=hypothesis_template, multi_label=False,
    )
    return result["labels"][0], float(result["scores"][0])

## 10. Run the pipeline over the sample

Stages 2–5 applied to every case in `sample_df` — this is the slowest cell
in the notebook, dominated by Stage 5b's zero-shot calls (Section 9).
Measured across 8 real, varied-length cases on one CPU-only laptop (after
the Section 9 fix): **~19s/case on average** (range 1–57s, depending on how
many relation candidates a case generates), i.e. roughly **~16 minutes for
the default `SAMPLE_SIZE = 50`**. That's still the single largest time
sink in the notebook — watch the `tqdm` rate on the first ~10 cases before
raising `SAMPLE_SIZE`, and treat the figure above as an order-of-magnitude
guide, not a promise, since it depends on this specific CPU and on how
entity-dense your sampled cases happen to be.

In [11]:
all_entities = []      # one row per accepted entity mention
all_qualifiers = []    # one row per qualifier mention (Dosage, Severity, ...)
all_triples = []       # one row per accepted (subject, relation, object) candidate

for _, case in tqdm(sample_df.iterrows(), total=len(sample_df), desc="cases"):
    entities, qualifiers = run_ner(case.case_text)
    annotated, doc = annotate_entities(case.case_text, entities)

    for e in annotated:
        all_entities.append({
            **{k: v for k, v in e.items() if k != "spacy_span"},
            "case_id": case.case_id,
        })
    for q in qualifiers:
        all_qualifiers.append({**q, "case_id": case.case_id})

    for subj, obj, sentence, labels in generate_relation_candidates(doc, annotated):
        relation, score = type_relation(sentence, subj["text"], obj["text"], labels)
        if score < RELATION_TYPE_THRESHOLD:
            continue
        all_triples.append({
            "case_id": case.case_id,
            "subject_text": subj["text"], "subject_start": subj["start"],
            "object_text": obj["text"], "object_start": obj["start"],
            "relation": relation, "score": score, "sentence": sentence,
            "subject_cui": subj["cui"], "object_cui": obj["cui"],
        })

entities_df = pd.DataFrame(all_entities)
qualifiers_df = pd.DataFrame(all_qualifiers)
triples_df = pd.DataFrame(all_triples)

print(f"{len(entities_df)} entity mentions, {len(qualifiers_df)} qualifier mentions, "
      f"{len(triples_df)} relation candidates above the typing threshold.")

cases: 100%|██████████| 50/50 [16:56<00:00, 20.33s/it]

2094 entity mentions, 226 qualifier mentions, 544 relation candidates above the typing threshold.


## 11. Stage 6 — Cross-case entity resolution

Entities sharing a UMLS CUI collapse into one node automatically. Entities
without a CUI (UMLS linking off, or no confident match) are clustered
*within their category* by SapBERT embedding similarity instead, using
agglomerative clustering with a cosine-distance threshold — e.g. unifying
"myocardial infarction" mentioned in one case with "heart attack" in
another.

In [ ]:
def resolve_entities(entities_df):
    entities_df = entities_df.copy()
    entities_df["node_id"] = None

    grounded = entities_df["cui"].notna()
    entities_df.loc[grounded, "node_id"] = "CUI:" + entities_df.loc[grounded, "cui"]

    for category, group in entities_df.loc[~grounded].groupby("category"):
        texts = group["text"].str.lower().tolist()
        if len(set(texts)) == 1:
            labels = np.zeros(len(texts), dtype=int)
        else:
            embeddings = embed_texts(texts)
            clustering = AgglomerativeClustering(
                n_clusters=None, distance_threshold=1 - ENTITY_RESOLUTION_SIM_THRESHOLD,
                metric="cosine", linkage="average",
            )
            labels = clustering.fit_predict(embeddings)
        entities_df.loc[group.index, "node_id"] = [
            f"CLUSTER:{category}:{label}" for label in labels
        ]

    return entities_df


entities_df = resolve_entities(entities_df)

# One display name per node: prefer the UMLS canonical name, then the most
# frequent surface form seen for that node.
node_labels = (
    entities_df.assign(display_name=entities_df["canonical_name"].fillna(entities_df["text"]))
    .groupby("node_id")["display_name"]
    .agg(lambda s: s.value_counts().idxmax())
)
node_categories = entities_df.groupby("node_id")["category"].first()
node_cui = entities_df.groupby("node_id")["cui"].first()

print(f"{entities_df['node_id'].nunique()} distinct nodes resolved from {len(entities_df)} mentions.")

## 12. Stage 7 — Faithfulness filter

Applies the stricter `FAITHFULNESS_THRESHOLD` to the same entailment score
computed during typing (Section 9) — no second model call, just a stricter
cutoff on a score already in hand. Then remaps each surviving triple's
subject/object to their resolved node ids (Section 11) so edges connect
graph nodes, not raw text spans.

In [13]:
# (case_id, start) -> node_id, built from the resolved entities.
node_lookup = entities_df.set_index(["case_id", "start"])["node_id"].to_dict()

accepted = triples_df[triples_df["score"] >= FAITHFULNESS_THRESHOLD].copy()

accepted["subject_node"] = accepted.apply(
    lambda r: node_lookup.get((r["case_id"], r["subject_start"])), axis=1
)
accepted["object_node"] = accepted.apply(
    lambda r: node_lookup.get((r["case_id"], r["object_start"])), axis=1
)
accepted = accepted.dropna(subset=["subject_node", "object_node"])

print(f"{len(accepted)} / {len(triples_df)} candidate triples survive the "
      f"faithfulness threshold ({FAITHFULNESS_THRESHOLD}) and resolve to graph nodes.")

430 / 544 candidate triples survive the faithfulness threshold (0.6) and resolve to graph nodes.


## 13. Stage 8 — Build and visualize the graph

In [ ]:
G = nx.MultiDiGraph()

for node_id in node_labels.index:
    G.add_node(
        node_id, label=node_labels[node_id], category=node_categories[node_id],
        node_type="entity", cui=node_cui[node_id],
    )

for _, row in accepted.iterrows():
    G.add_edge(
        row["subject_node"], row["object_node"],
        relation=row["relation"], score=row["score"],
        case_id=row["case_id"], sentence=row["sentence"],
        edge_type="relation",
    )

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")

In [15]:
from pyvis.network import Network

CATEGORY_COLORS = {
    "Symptom": "#e07a5f", "Disease": "#3d405b", "Drug": "#81b29a",
    "Laboratory Test": "#f2cc8f", "Anatomy": "#a8dadc", "Procedure": "#e9c46a",
}

net = Network(notebook=True, cdn_resources="in_line", height="750px", width="100%", directed=True)
for node_id, data in G.nodes(data=True):
    net.add_node(
        node_id, label=data["label"], title=f"{data['category']} ({node_id})",
        color=CATEGORY_COLORS.get(data["category"], "#cccccc"),
    )
for u, v, data in G.edges(data=True):
    net.add_edge(u, v, label=data["relation"], title=f"score={data['score']:.2f} · case={data['case_id']}")

preview_path = OUTPUT_DIR / "kg_preview.html"
net.show(str(preview_path))
print(f"Interactive preview written to {preview_path}")

../data/kg_extraction/kg_preview.html
Interactive preview written to ../data/kg_extraction/kg_preview.html


## 14. Optional — load into local Neo4j

Skips gracefully if Neo4j isn't reachable — no error, just a printed note.
Start it with `docker compose --profile graph up neo4j` (see
`../README.md`) before running this cell if you want the interactive
Cypher demo.

In [16]:
NEO4J_URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "localdevpassword"   # matches docker-compose.yml's neo4j service — local dev only

try:
    from neo4j import GraphDatabase
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    neo4j_available = True
except Exception as exc:
    print(f"Neo4j not reachable at {NEO4J_URI} ({exc!r}) — skipping. "
          "Start it with `docker compose --profile graph up neo4j` if you want this step.")
    neo4j_available = False

if neo4j_available:
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
        for node_id, data in G.nodes(data=True):
            session.run(
                "MERGE (n:Entity {id: $id}) SET n.label = $label, n.category = $category",
                id=node_id, label=data["label"], category=data["category"],
            )
        for u, v, data in G.edges(data=True):
            session.run(
                "MATCH (a:Entity {id: $u}), (b:Entity {id: $v}) "
                "MERGE (a)-[r:RELATION {type: $rel, case_id: $case_id}]->(b) "
                "SET r.score = $score",
                u=u, v=v, rel=data["relation"], case_id=data["case_id"], score=data["score"],
            )
    driver.close()
    print(f"Graph loaded into Neo4j — open http://localhost:7474 "
          f"(auth {NEO4J_USER}/{NEO4J_PASSWORD}) to explore with Cypher.")

Neo4j not reachable at bolt://localhost:7687 (ServiceUnavailable("Couldn't connect to localhost:7687 (resolved to ('127.0.0.1:7687',)):\nFailed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [Errno 111] Connection refused)")) — skipping. Start it with `docker compose --profile graph up neo4j` if you want this step.


## 15. Quick quality check

No gold KG exists for this corpus (see `docs/data_source.md`), so this is a
sanity check, not an evaluation. Section 7 of the methodology doc proposes
hand-annotating a small pilot sample to properly calibrate
`RELATION_TYPE_THRESHOLD` / `FAITHFULNESS_THRESHOLD` and measure
precision/recall — that step is manual and intentionally not automated
here.

In [17]:
print("Entities by category:")
display(entities_df["category"].value_counts())

print("\nRelations by type:")
display(accepted["relation"].value_counts())

print("\nSample of accepted triples:")
display(accepted[["case_id", "subject_text", "relation", "object_text", "score"]].sample(
    min(10, len(accepted)), random_state=RANDOM_SEED
))

Entities by category:


category
Procedure          703
Symptom            489
Anatomy            378
Laboratory Test    284
Disease            135
Drug               105
Name: count, dtype: int64


Relations by type:


relation
reveals              199
located_in           129
underwent             42
has_lab_result        28
treats                10
diagnosed_with         8
family_history_of      5
ruled_out              5
has_symptom            4
Name: count, dtype: int64


Sample of accepted triples:


,case_id,subject_text,relation,object_text,score
535,PMC7502135_01,biopsies,reveals,4,0.727395
110,PMC9189279_01,serum creatinine,has_lab_result,89,0.665628
239,PMC7446910_01,titers,reveals,elevated,0.705988
40,PMC12426244_02,magnetic resonance imaging,reveals,worsened,0.761965
465,PMC7362985_01,aspiration,reveals,aden,0.678715
485,PMC10354338_01,##ac,located_in,left ventricular,0.784402
212,PMC9609878_01,tube voltage,located_in,antecubital,0.721547
209,PMC9609878_01,ct,reveals,kv,0.795579
223,PMC9609878_01,lesions,located_in,tracheal,0.946498
237,PMC7446910_01,##ievirus,has_lab_result,elevated,0.953923


## 16. Export

Written to `../data/kg_extraction/` (gitignored, like the rest of `data/`).

**Format: CSV, not Parquet.** An earlier version of this notebook wrote
Parquet — not because the input required it (the input is DuckDB, read via
SQL, not a Parquet file), but just as an unexamined default matching the
Parquet files `01_data_preparation.ipynb` downloads from Zenodo. Nothing
about *these* outputs needs Parquet's columnar/typed storage: they're at
most a few thousand rows, meant for pandas, spreadsheet, or graph-tool
consumption where CSV's universal support matters more than Parquet's
size/dtype advantages. So: CSV throughout, `index=False`.

**Two output shapes:**
- `entities.csv`, `qualifiers.csv`, `triples.csv` — the raw per-mention /
  per-candidate tables from Sections 7–12, for pandas-side inspection
  (e.g. comparing the pipeline's triples for one case against a group's
  manual annotation, Steps 3–5 of the activity plan).
- `nodes.csv`, `edges.csv` — a graph interchange pair built from `G`, with
  **case nodes and case→entity "mentions" edges added** (`G` itself, used
  by the Section 13 pyvis preview, stays entity-only). Column names
  (`source`, `target`, `interaction`) match Cytoscape's default simple
  network table so it auto-detects them: **File → Import → Network from
  File → `edges.csv`**, then **File → Import → Table from File →
  `nodes.csv`** (import as a node table, key column `id`, matched against
  the network's "shared name"). `kg.graphml` (from the same case-inclusive
  graph) is also written, and Cytoscape can import that directly too, if
  you'd rather skip the two-step CSV import.

**`graph_data.js`** — the same nodes/edges, as a JS object literal, for the
dedicated viewer at [`../viewer/graph_viewer.html`](../viewer/graph_viewer.html)
(versioned, not regenerated by this notebook — open it after running this
cell to explore the graph with the case-node toggle and click-to-highlight/
filter described there).

In [ ]:
import json

CASE_NODE_PREFIX = "CASE:"


def build_export_graph(G, entities_df, sample_df):
    """G plus one node per case and a `mentions` edge to each entity it
    contributed (see Section 16 markdown). Kept separate from `G` itself
    so the Section 13 pyvis preview stays entity-only."""
    G_export = G.copy()

    case_info = sample_df.set_index("case_id")[["age", "gender"]]
    mention_counts = (
        entities_df.groupby(["case_id", "node_id"]).size().rename("mention_count").reset_index()
    )

    for case_id in entities_df["case_id"].unique():
        age = case_info.loc[case_id, "age"] if case_id in case_info.index else None
        gender = case_info.loc[case_id, "gender"] if case_id in case_info.index else None
        G_export.add_node(
            f"{CASE_NODE_PREFIX}{case_id}",
            label=case_id, category="Case", node_type="case", cui=None,
            age=None if pd.isna(age) else float(age), gender=gender,
        )

    for _, row in mention_counts.iterrows():
        G_export.add_edge(
            f"{CASE_NODE_PREFIX}{row['case_id']}", row["node_id"],
            relation="mentions", edge_type="mentions", score=None,
            case_id=row["case_id"], sentence=None, mention_count=int(row["mention_count"]),
        )

    return G_export


G_export = build_export_graph(G, entities_df, sample_df)
n_case_nodes = sum(1 for _, d in G_export.nodes(data=True) if d.get("node_type") == "case")
print(f"Export graph: {G_export.number_of_nodes()} nodes ({n_case_nodes} case nodes), "
      f"{G_export.number_of_edges()} edges "
      f"({G_export.number_of_edges() - G.number_of_edges()} of them 'mentions' edges).")

# --- raw tables (CSV — see Section 16 markdown for why, not Parquet) ---
entities_df.drop(columns=["spacy_span"], errors="ignore").to_csv(OUTPUT_DIR / "entities.csv", index=False)
qualifiers_df.to_csv(OUTPUT_DIR / "qualifiers.csv", index=False)
accepted.to_csv(OUTPUT_DIR / "triples.csv", index=False)

# --- graph interchange: Cytoscape-friendly column names -----------------
nodes_records = [
    {
        "id": node_id,
        "label": data.get("label", node_id),
        "category": data.get("category", "Unknown"),
        "node_type": data.get("node_type", "entity"),
        "cui": data.get("cui"),
        "age": data.get("age"),
        "gender": data.get("gender"),
    }
    for node_id, data in G_export.nodes(data=True)
]
edges_records = [
    {
        "source": u,
        "target": v,
        "interaction": data.get("relation"),
        "edge_type": data.get("edge_type"),
        "score": data.get("score"),
        "case_id": data.get("case_id"),
        "sentence": data.get("sentence"),
        "mention_count": data.get("mention_count"),
    }
    for u, v, data in G_export.edges(data=True)
]

pd.DataFrame(nodes_records).to_csv(OUTPUT_DIR / "nodes.csv", index=False)
pd.DataFrame(edges_records).to_csv(OUTPUT_DIR / "edges.csv", index=False)


def drop_none_attrs(graph):
    # GraphML has no null/None type — write_graphml raises on any attribute
    # whose value is None (case nodes without an age, mentions edges'
    # unused score/sentence fields, ...). CSV/JSON both handle None fine,
    # so this is only needed for the graphml export below.
    graph = graph.copy()
    for _, data in graph.nodes(data=True):
        for key in [k for k, v in data.items() if v is None]:
            del data[key]
    for _, _, data in graph.edges(data=True):
        for key in [k for k, v in data.items() if v is None]:
            del data[key]
    return graph


nx.write_graphml(drop_none_attrs(G_export), OUTPUT_DIR / "kg.graphml")


# --- viewer data: ../viewer/graph_viewer.html loads this via <script> --
def _json_safe(value):
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return None if np.isnan(value) else float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value


def to_jsonable(records):
    return [{k: _json_safe(v) for k, v in record.items()} for record in records]


graph_data = {"nodes": to_jsonable(nodes_records), "edges": to_jsonable(edges_records)}
with open(OUTPUT_DIR / "graph_data.js", "w") as f:
    f.write("// Auto-generated by 02_kg_extraction.ipynb — do not edit by hand.\n")
    f.write("const GRAPH_DATA = ")
    json.dump(graph_data, f, indent=2)
    f.write(";\n")

print(f"Exported entities/qualifiers/triples (CSV), nodes.csv, edges.csv, kg.graphml, "
      f"and graph_data.js to {OUTPUT_DIR}/")

## Next steps

- **Explore the graph in [`../viewer/graph_viewer.html`](../viewer/graph_viewer.html).**
  Open it after running Section 16 — it auto-loads `graph_data.js` from
  this run. Toggle case nodes on to see which entities came from which
  case, and click a case node to highlight or filter down to everything
  it directly or indirectly points to.
- **Calibrate thresholds against a pilot gold sample** (methodology doc,
  Section 7): hand-annotate ~20 cases using the LREC 2020 scheme and sweep
  `RELATION_TYPE_THRESHOLD` / `FAITHFULNESS_THRESHOLD` against it.
- **Use this as Step 0** of the activity plan: show the viewer or Neo4j
  graph in class before Step 1, then revisit specific cases' triples during
  Steps 3–5 as a discussion prompt for manual annotation.
- **If relation quality isn't good enough after trying this:** fine-tune a
  small BioClinicalBERT relation classifier on the same pilot annotation
  set (methodology doc, Section 5, Stage 5) — reuses annotation effort
  already being done for evaluation.